In [1]:
import os
from pathlib import Path
from src.utils import *
from src.config import PROCESSED_IMG_DIR, RAW_IMG_DIR
from src.feature_extraction import *
from src.feature_extraction import _create_fixed_box
from src.region_visualization import collect_all_rois, build_mosaic, show_mosaic
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
from src.config import ROOT_DIR
import time
import cv2
import random
from multiprocessing import Pool, cpu_count
from functools import partial
from src.feature_extraction import _extract_features_for_image

In [2]:
paths = get_all_image_paths(directory = PROCESSED_IMG_DIR)
separated_paths = seprate_processed_files(paths) # Separate all paths that end in .png
processed_paths = separated_paths[0] # These are the processed images (paths only)
masks_paths = separated_paths[1]     # These are the binary masks (paths only)
skeletons_paths = separated_paths[2] # These are the skeletons (paths only)
xml_paths = []                       # And this for the .xml (paths only)
ground_truth_bboxes = []             # And this for the ground truth bboxes
for img_path in processed_paths:
    xml_path = get_xml_path(img_path)
    xml_path = xml_path.replace('_processed.xml', '_bbox.xml') # Fix the ending from _processed.xml to _bbox.xml
    xml_paths.append(xml_path)
    bbox = parse_stenosis_xml(xml_path)
    ground_truth_bboxes.append(bbox)
print('XML files: ', len(xml_paths))
print('Ground truth bounding boxes: ', len(ground_truth_bboxes))

STAT_NAMES  = ['mean', 'var', 'entropy', 'energy', 'kurtosis', 'skewness']
TILE_LABELS = ['tl', 'tr', 'bl', 'br']
ROI_SIZE    = 80
HALF_SIZE   = ROI_SIZE // 2
LABELING_THRESHOLD = 0.50
N_SIZES = 6
N_ORIENTATIONS = 12

Total images found: 24975
Processed images: 8325, Masks: 8325, Skeletons: 8325
XML files:  8325
Ground truth bounding boxes:  8325


In [14]:
# ─────────────────────────────────────────────────────────────
# CELL A: Extract ROI coordinates → roi_coordinates.csv
# ─────────────────────────────────────────────────────────────
OUTPUT_DIR_CSV = Path(os.path.join(ROOT_DIR, "notebooks/csv_files/all_images"))

full_dataset = list(zip(processed_paths, masks_paths, skeletons_paths, ground_truth_bboxes))

rows_coords = []   # roi_name, image_name, xmin, ymin, xmax, ymax, label

print(f"Starting coordinate extraction over {len(full_dataset)} images...")
start = time.time()

for img_path, mask_path, skel_path, gt_boxes in full_dataset:

    img      = load_image(img_path)
    skeleton = load_image(skel_path)
    if img is None or skeleton is None:
        continue

    img_shape = img.shape[:2]

    p          = Path(img_path)
    patient_id = p.parts[-3]
    serie_id   = p.parts[-2]
    frame_stem = p.stem.replace('_processed', '')[-4:]
    image_name = f"{patient_id}_{serie_id}_{frame_stem}"
    print("Extracting ROIs from image ", image_name)
    sampled_boxes  = extract_rois_ferran(skeleton, img_shape, roi_size=ROI_SIZE, total_rois=100)
    sampled_labels = label_rois(sampled_boxes, gt_boxes, threshold=LABELING_THRESHOLD)

    all_boxes_and_labels = list(zip(sampled_boxes, sampled_labels))

    for gt_box in gt_boxes:
        cx = (gt_box['xmin'] + gt_box['xmax']) // 2
        cy = (gt_box['ymin'] + gt_box['ymax']) // 2
        fixed = _create_fixed_box(cx, cy, HALF_SIZE, img_shape)
        all_boxes_and_labels.append((fixed, 1))

    for roi_idx, (box, lbl) in enumerate(all_boxes_and_labels, start=1):
        rows_coords.append({
            'roi_name'   : f"{image_name}_{roi_idx}",
            'image_name' : image_name,
            'xmin'       : box['xmin'],
            'ymin'       : box['ymin'],
            'xmax'       : box['xmax'],
            'ymax'       : box['ymax'],
            'label'      : lbl,
        })

elapsed = time.time() - start
df_coords = pd.DataFrame(rows_coords)

print(f"\nDone in {elapsed:.1f}s — {len(rows_coords)} ROIs extracted.")
print(f"df_coords shape : {df_coords.shape}")
print(f"Stenosis ROIs   : {df_coords['label'].sum()}")
print(f"Healthy ROIs    : {(df_coords['label'] == 0).sum()}")

OUTPUT_DIR_CSV.mkdir(parents=True, exist_ok=True)
coords_csv_path = OUTPUT_DIR_CSV / "roi_coordinates.csv"
df_coords.to_csv(coords_csv_path, index=False)
print(f"\nSaved: {coords_csv_path}")

Starting coordinate extraction over 8325 images...
Extracting ROIs from image  002_5_0016
Extracting ROIs from image  002_5_0017
Extracting ROIs from image  002_5_0018
Extracting ROIs from image  002_5_0019
Extracting ROIs from image  002_5_0020
Extracting ROIs from image  002_5_0021
Extracting ROIs from image  002_5_0022
Extracting ROIs from image  002_5_0023
Extracting ROIs from image  002_5_0024
Extracting ROIs from image  002_5_0025
Extracting ROIs from image  002_5_0026
Extracting ROIs from image  002_5_0027
Extracting ROIs from image  002_5_0028
Extracting ROIs from image  002_5_0029
Extracting ROIs from image  002_5_0030
Extracting ROIs from image  002_5_0031
Extracting ROIs from image  002_5_0032
Extracting ROIs from image  002_5_0033
Extracting ROIs from image  002_5_0034
Extracting ROIs from image  002_5_0035
Extracting ROIs from image  002_5_0036
Extracting ROIs from image  002_5_0037
Extracting ROIs from image  002_5_0038
Extracting ROIs from image  002_5_0039
Extracting RO

OSError: [WinError 123] The filename, directory name, or volume label syntax is incorrect: 'C:\\Users\\ferra\\MIC\\1r_any_UNICAS\\2n_Semestre\\Image_Processing_and_Analysis\\project\\MIC_project\\Proposal_StenosisDetection\\notebooks\\csv_files\x07ll_images'

In [16]:
OUTPUT_DIR_CSV = Path(os.path.join(ROOT_DIR, "notebooks/csv_files/all_images"))
OUTPUT_DIR_CSV.mkdir(parents=True, exist_ok=True)
coords_csv_path = OUTPUT_DIR_CSV / "roi_coordinates.csv"
df_coords.to_csv(coords_csv_path, index=False)
print(f"\nSaved: {coords_csv_path}")


Saved: C:\Users\ferra\MIC\1r_any_UNICAS\2n_Semestre\Image_Processing_and_Analysis\project\MIC_project\Proposal_StenosisDetection\notebooks\csv_files\all_images\roi_coordinates.csv


In [17]:
# ─────────────────────────────────────────────────────────────
# SENSITIVITY ANALYSIS
# Only the first 100 ROIs per image are pipeline-proposed.
# The manually-centred GT ROIs appended afterwards must be
# excluded so they don't inflate the sensitivity estimate.
# ─────────────────────────────────────────────────────────────
df_coords = pd.read_csv(OUTPUT_DIR_CSV / "roi_coordinates.csv")

# Keep only the pipeline-proposed ROIs (first 100 per image)
df_pipeline = (
    df_coords
    .groupby('image_name', sort=False)
    .head(100)          # first 100 rows per image = the sampled ROIs
    .reset_index(drop=True)
)

# Per-image positive count (pipeline ROIs only)
pos_per_image = (
    df_pipeline
    .groupby('image_name')['label']
    .sum()              # number of positive ROIs per image
)

n_total_images   = len(pos_per_image)
n_detected       = (pos_per_image >= 1).sum()   # images with ≥1 positive ROI
sensitivity      = n_detected / n_total_images if n_total_images > 0 else 0.0
avg_pos_per_img  = pos_per_image.mean()
avg_pos_detected = pos_per_image[pos_per_image >= 1].mean()  # among detected only

print("\n" + "═" * 52)
print("  PIPELINE SENSITIVITY REPORT")
print("═" * 52)
print(f"  Total annotated images            : {n_total_images}")
print(f"  Images with ≥1 positive ROI       : {n_detected}")
print(f"  Images with 0 positive ROIs       : {n_total_images - n_detected}")
print(f"  Sensitivity                       : {sensitivity:.4f}  ({sensitivity*100:.2f}%)")
print("─" * 52)
print(f"  Avg positive ROIs / image (all)   : {avg_pos_per_img:.3f}")
print(f"  Avg positive ROIs / image (detected only): {avg_pos_detected:.3f}")
print("═" * 52)


════════════════════════════════════════════════════
  PIPELINE SENSITIVITY REPORT
════════════════════════════════════════════════════
  Total annotated images            : 8325
  Images with ≥1 positive ROI       : 7980
  Images with 0 positive ROIs       : 345
  Sensitivity                       : 0.9586  (95.86%)
────────────────────────────────────────────────────
  Avg positive ROIs / image (all)   : 4.315
  Avg positive ROIs / image (detected only): 4.502
════════════════════════════════════════════════════


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL B: Extract features for each ROI → roi_features.csv
# Parallelised across all available CPU cores.
# Requires df_coords from Cell A to be in memory.
# ─────────────────────────────────────────────────────────────

# ── Run parallel extraction ───────────────────────────────────

gabor_bank = build_gabor_bank(ksize=31, n_sizes=N_SIZES, n_orientations=N_ORIENTATIONS)
feature_cols = build_feature_column_names(
    n_filters=len(gabor_bank),
    stat_names=STAT_NAMES,
    tile_labels=TILE_LABELS,
)

worker_fn = partial(
    _extract_features_for_image,
    roi_size          = ROI_SIZE,
    half_size         = HALF_SIZE,
    labeling_threshold= LABELING_THRESHOLD,
    gabor_bank        = gabor_bank,
    feature_cols      = feature_cols,
)

n_cores = cpu_count()
print(f"Starting parallel feature extraction over {len(full_dataset)} images using {n_cores} cores...")
start = time.time()

with Pool(processes=n_cores) as pool:
    results = pool.map(worker_fn, full_dataset)

# Flatten the list of lists
rows_features = [row for image_rows in results for row in image_rows]

elapsed = time.time() - start
df_features = pd.DataFrame(rows_features)

print(f"\nDone in {elapsed:.1f}s — {len(rows_features)} ROIs extracted.")
print(f"df_features shape : {df_features.shape}")

features_csv_path = OUTPUT_DIR_CSV / "roi_features.csv"
df_features.to_csv(features_csv_path, index=False)
print(f"\nSaved: {features_csv_path}")

## EXPLORATION OF EXTRACTED FEATURES

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPLORATION: Load CSVs and reconstruct feature matrix
# ─────────────────────────────────────────────────────────────
df_features_all = pd.read_csv(OUTPUT_DIR_CSV / "roi_features.csv")

y_all        = df_features_all['label'].values
X_all        = df_features_all.drop(columns=['roi_name', 'label']).values
feature_cols_all = [c for c in df_features_all.columns if c not in ('roi_name', 'label')]

scaler_all   = StandardScaler()
X_scaled_all = scaler_all.fit_transform(X_all)

print(f"Loaded feature matrix : {X_all.shape}")
print(f"Stenosis ROIs         : {np.sum(y_all)}")
print(f"Healthy ROIs          : {len(y_all) - np.sum(y_all)}")
print(f"Class imbalance       : {round((len(y_all) - np.sum(y_all)) / np.sum(y_all), 2)}× more healthy than stenosis")

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPLORATION: 3D PCA scatter plot
# ─────────────────────────────────────────────────────────────
%matplotlib tk

pca_3d_all   = PCA(n_components=3)
X_pca_3d_all = pca_3d_all.fit_transform(X_scaled_all)
total_variance_all = pca_3d_all.explained_variance_ratio_.sum() * 100

fig = plt.figure(figsize=(10, 8))
ax  = fig.add_subplot(111, projection='3d')

ax.scatter(X_pca_3d_all[y_all == 0, 0], X_pca_3d_all[y_all == 0, 1], X_pca_3d_all[y_all == 0, 2],
           alpha=0.4, label='Healthy (0)',  color='#3498db', s=25)
ax.scatter(X_pca_3d_all[y_all == 1, 0], X_pca_3d_all[y_all == 1, 1], X_pca_3d_all[y_all == 1, 2],
           alpha=0.8, label='Stenosis (1)', color='#e74c3c', marker='x', s=45)

ax.set_title(f"3D PCA — All Images\n(Total Explained Variance: {total_variance_all:.1f}%)",
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel(f"PC 1 ({pca_3d_all.explained_variance_ratio_[0]*100:.1f}%)", fontsize=10, labelpad=10)
ax.set_ylabel(f"PC 2 ({pca_3d_all.explained_variance_ratio_[1]*100:.1f}%)", fontsize=10, labelpad=10)
ax.set_zlabel(f"PC 3 ({pca_3d_all.explained_variance_ratio_[2]*100:.1f}%)", fontsize=10, labelpad=10)
ax.xaxis.set_pane_color((0.95, 0.95, 0.95, 1.0))
ax.yaxis.set_pane_color((0.95, 0.95, 0.95, 1.0))
ax.zaxis.set_pane_color((0.95, 0.95, 0.95, 1.0))
ax.view_init(elev=25, azim=45)
ax.legend(loc='upper left', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPLORATION: Feature ranking and boxplots
# ─────────────────────────────────────────────────────────────
%matplotlib inline

f_values_all, p_values_all = f_classif(X_scaled_all, y_all)

n_indeces = 8
top_n_indices_all = np.argsort(f_values_all)[::-1][:n_indeces]

print(f"\n--- TOP {n_indeces} MOST DISCRIMINATIVE FEATURES — All Images ---")
for rank, idx in enumerate(top_n_indices_all, start=1):
    print(f"Rank {rank}: Feature {idx:4d}  '{feature_cols_all[idx]}'  — F-score: {f_values_all[idx]:.2f}  p: {p_values_all[idx]:.2e}")

labels_all = np.where(y_all == 0, 'Healthy', 'Stenosis')

fig, axes = plt.subplots(1, n_indeces, figsize=(16, 5))
fig.suptitle(f"Top {n_indeces} Most Discriminative Features — All Images", fontsize=16)

for i, feat_idx in enumerate(top_n_indices_all):
    ax = axes[i]
    sns.boxplot(
        x=labels_all,
        y=X_scaled_all[:, feat_idx],
        hue=labels_all,
        order=['Healthy', 'Stenosis'],
        hue_order=['Healthy', 'Stenosis'],
        palette={'Healthy': '#3498db', 'Stenosis': '#e74c3c'},
        legend=False,
        ax=ax,
    )
    ax.set_title(feature_cols_all[feat_idx], fontsize=7, wrap=True)
    ax.set_xlabel("")
    ax.set_ylabel("Standardised Value" if i == 0 else "")

plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPLORATION: Scree plot
# ─────────────────────────────────────────────────────────────
%matplotlib inline

pca_full_all = PCA().fit(X_scaled_all)
cumvar_all   = np.cumsum(pca_full_all.explained_variance_ratio_) * 100
n_components_all = np.arange(1, len(cumvar_all) + 1)

thresh_80_all = np.searchsorted(cumvar_all, 80) + 1
thresh_90_all = np.searchsorted(cumvar_all, 90) + 1
thresh_95_all = np.searchsorted(cumvar_all, 95) + 1

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(n_components_all, cumvar_all, color='#3498db', linewidth=1.5, label='Cumulative explained variance')
ax.fill_between(n_components_all, cumvar_all, alpha=0.1, color='#3498db')

for thresh, n, color in zip([80, 90, 95],
                             [thresh_80_all, thresh_90_all, thresh_95_all],
                             ['#2ecc71', '#e67e22', '#e74c3c']):
    ax.axhline(thresh, color=color, linestyle='--', linewidth=1, alpha=0.7)
    ax.axvline(n,      color=color, linestyle='--', linewidth=1, alpha=0.7)
    ax.annotate(f"{thresh}% → {n} components",
                xy=(n, thresh),
                xytext=(n + len(n_components_all) * 0.02, thresh - 4),
                fontsize=9, color=color)

ax.set_title("Scree Plot — Cumulative Explained Variance (All Images)", fontsize=14, fontweight='bold')
ax.set_xlabel("Number of Components", fontsize=11)
ax.set_ylabel("Cumulative Explained Variance (%)", fontsize=11)
ax.set_xlim(1, len(n_components_all))
ax.set_ylim(0, 101)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"Total features      : {X_scaled_all.shape[1]}")
print(f"Components for 80%  : {thresh_80_all}  ({thresh_80_all / X_scaled_all.shape[1] * 100:.1f}% of features)")
print(f"Components for 90%  : {thresh_90_all}  ({thresh_90_all / X_scaled_all.shape[1] * 100:.1f}% of features)")
print(f"Components for 95%  : {thresh_95_all}  ({thresh_95_all / X_scaled_all.shape[1] * 100:.1f}% of features)")